# Notebook 6 — Shared head or exposure? (MOSEI, frozen M4, CPU)

**Question.** Notebook 5 showed that M4's representation still contains the audio/video information when text is missing,
and a new head trained *only* on that condition recovers it (video +2.94 pp, audio+video +1.39 pp). That head saw the
condition 100 % of the time; M4's shared head saw it in about 7 % (video only) / 15 % (audio+video) of training samples.
So which is it?
- **Exposure:** a single shared head is enough if every modality combination is seen often enough, or
- **Conditional decision boundaries:** different combinations need different classifiers, even with equal exposure.

**Heads compared** (all on M4's frozen classifier input, i.e. the fused vector for each modality combination; same
architecture as M4's own classifier; same optimiser and early stopping as every v2 model):
| Head | What it is |
|---|---|
| M4 own | M4's original classifier (reference) |
| H1 | one shared head, retrained with **M4's original exposure** (modality dropout 0.3) |
| H2 | one shared head, **every combination equally often** |
| H3 | H2 + a tiny per-combination adjustment (FiLM: 2 × 128 numbers per combination, starting at zero) |
| H4 | a **separate head for each combination**, trained only on it |

**Workload (counted before running):** representation extraction = 3 seeds × 7 combinations × 3 splits = 63 inference passes
of the saved M4; head trainings = 3 shared variants × 3 seeds + 7 separate heads × 3 seeds = **30** small trainings
(run 4 at a time). A benchmark on the first seed is printed before the rest runs. Everything is saved as it finishes;
after a restart, **Run all** continues where it stopped. Nothing from earlier notebooks is modified.

**Decision rule (fixed in advance).** Primary conditions: video only and audio+video (text missing). Margin 1.0 pp macro-F1;
"significant" = seed+video bootstrap 95 % interval excludes 0.
- **H4 − H2 > +1 pp and significant** (in either primary condition) → combinations need their own decision boundaries beyond
  exposure → a conditional-classifier direction is justified, to be compared with missing-aware prompts / MPLMM / Flex-MoE.
- otherwise → **exposure explains it**: equal exposure with one shared head is enough (close to MCE's availability
  reweighting) → no new classifier mechanism is justified; report it as an analysis finding.
- **Guardrail:** a variant that loses more than 1 pp on the full condition (T+A+V) versus M4 own is flagged.
- H2 − H1 (effect of exposure alone) and H3 − H2 (does a tiny adjustment recover H4?) are reported as secondary results.

**Status:** exploratory (MOSEI test set reused). Confirmation on MOSI follows with the same settings, no tuning.

In [1]:
import os, sys, time, json, random, pickle, platform, hashlib, itertools
IN_COLAB = "google.colab" in sys.modules
TEST_MODE = os.environ.get("EMOFAR_TEST") == "1"
if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
import numpy as np, pandas as pd, torch, torch.nn as nn, torch.nn.functional as F
import sklearn
from joblib import Parallel, delayed
DEVICE = "cpu"
torch.backends.cudnn.deterministic = True

Mounted at /content/drive


## 1. Paths and settings

In [2]:
BASE_OUT = "/content/drive/MyDrive/emofar_results"
if TEST_MODE:
    BASE_OUT = os.environ["EMOFAR_OUT"]
V2_DIR, V4_DIR, V5_DIR = (f"{BASE_OUT}/{d}" for d in ["mosei_v2_cpu", "mosei_v4_capacity_utility", "mosei_v5_representation_diagnostics"])
OUT_DIR = f"{BASE_OUT}/mosei_v6_head_vs_exposure"
N_BOOT = 10000 if not TEST_MODE else 500
MARGIN_PP = 1.0
assert os.path.isdir(V2_DIR), f"missing folder: {V2_DIR}"
os.makedirs(f"{OUT_DIR}/reps", exist_ok=True); os.makedirs(f"{OUT_DIR}/heads", exist_ok=True)

v2_info = json.load(open(f"{V2_DIR}/run_info.json"))
SEEDS = v2_info["config"]["SEEDS"]
logs = [json.load(open(f"{V2_DIR}/checkpoints/{m}_seed{s}_log.json")) for m in ["M1", "M2", "M3", "M4"] for s in SEEDS]
cfg = logs[0]["train_config"]; assert all(l["train_config"] == cfg for l in logs)
DATA_PATH = cfg["DATA_PATH"] if not TEST_MODE else os.environ["EMOFAR_MOSI"]
D_MODEL, EPOCHS, PATIENCE, BATCH_SIZE, LR, MOD_DROPOUT = (cfg[k] for k in ["D_MODEL", "EPOCHS", "PATIENCE", "BATCH_SIZE", "LR", "MOD_DROPOUT"])
THREADS = int(v2_info["environment"]["torch_threads"]); torch.set_num_threads(THREADS)
N_JOBS = max(1, min(THREADS, os.cpu_count() or 1))
ENV = {"python": platform.python_version(), "torch": torch.__version__, "numpy": np.__version__, "sklearn": sklearn.__version__,
       "torch_threads": torch.get_num_threads(), "n_jobs": N_JOBS, "os": platform.platform(),
       "started_utc": time.strftime("%Y-%m-%d %H:%M:%S", time.gmtime())}
print(json.dumps(ENV, indent=1))
json.dump({"environment": ENV, "v2_train_config": cfg, "seeds": SEEDS, "n_boot": N_BOOT, "margin_pp": MARGIN_PP},
          open(f"{OUT_DIR}/run_info.json", "w"), indent=2)
RUN_DIR = OUT_DIR
CKPT_DIR = f"{V2_DIR}/checkpoints"
LABELS = ["negative", "neutral", "positive"]
MODS = ["text", "audio", "vision"]
CONDS = ["T", "A", "V", "A+V", "T+A", "T+V", "T+A+V"]
ROW = {"T": (1, 0, 0), "A": (0, 1, 0), "V": (0, 0, 1), "A+V": (0, 1, 1), "T+A": (1, 1, 0), "T+V": (1, 0, 1), "T+A+V": (1, 1, 1)}
COND_OF_ROW = {tuple(map(float, v)): i for i, v in enumerate(ROW[c] for c in CONDS)}

{
 "python": "3.13.18",
 "torch": "2.14.1+cpu",
 "numpy": "2.4.6",
 "sklearn": "1.6.1",
 "torch_threads": 4,
 "n_jobs": 2,
 "os": "Linux-6.6.159+-x86_64-with-glibc2.39",
 "started_utc": "2026-11-24 13:23:54"
}


## 2. Data (code copied unchanged from v2) and agreement with v2

In [3]:
t0 = time.time()
with open(DATA_PATH, "rb") as f:
    raw = pickle.load(f)
print(f"Loaded in {time.time() - t0:.0f}s")

def prep(split):
    d = raw[split]
    mask = d["text_bert"][:, 1, :].astype(bool)                        # (N, 50) real-word steps
    feats = {"text": d["text"].astype(np.float32),
             "audio": np.nan_to_num(d["audio"].astype(np.float32), nan=0.0, posinf=0.0, neginf=0.0),
             "vision": np.nan_to_num(d["vision"].astype(np.float32), nan=0.0, posinf=0.0, neginf=0.0)}
    score = d["regression_labels"].astype(np.float32)
    y = np.where(score < 0, 0, np.where(score == 0, 1, 2))
    return {"feats": feats, "mask": mask, "y": y, "score": score, "raw_text": d["raw_text"], "id": d["id"]}

data = {s: prep(s) for s in ["train", "valid", "test"]}
del raw

for m in ["audio", "vision"]:                                           # standardise with train statistics
    tr = data["train"]["feats"][m][data["train"]["mask"]]
    mu, sd = tr.mean(0), tr.std(0) + 1e-6
    for s in data:
        x = (data[s]["feats"][m] - mu) / sd
        data[s]["feats"][m] = np.clip(x, -10, 10) * data[s]["mask"][..., None]

DIMS = {m: data["train"]["feats"][m].shape[-1] for m in MODS}
print("Feature sizes:", DIMS)
for s in data:
    counts = np.bincount(data[s]["y"], minlength=3)
    print(f"{s:5s}: {len(data[s]['y']):6,} clips | " + " / ".join(f"{LABELS[i]} {counts[i]:,}" for i in range(3)))
np.save(f"{RUN_DIR}/test_labels.npy", data["test"]["y"]); np.save(f"{RUN_DIR}/valid_labels.npy", data["valid"]["y"])
np.save(f"{RUN_DIR}/test_scores.npy", data["test"]["score"])
pd.Series(data["test"]["id"]).to_csv(f"{RUN_DIR}/test_ids.csv", index=False, header=["id"])
v2_test_ids = pd.read_csv(f"{V2_DIR}/test_ids.csv", dtype=str)["id"].values
assert (v2_test_ids == np.asarray(data["test"]["id"]).astype(str)).all(), "test IDs/order differ from v2"
assert (np.load(f"{V2_DIR}/test_labels.npy") == data["test"]["y"]).all() and (np.load(f"{V2_DIR}/valid_labels.npy") == data["valid"]["y"]).all()
assert len(data["train"]["y"]) == cfg["n_train"]
print("Data match the v2 artifacts exactly.")

Loaded in 93s
Feature sizes: {'text': 768, 'audio': 74, 'vision': 35}
train: 16,326 clips | negative 4,738 / neutral 3,540 / positive 8,048
valid:  1,871 clips | negative 506 / neutral 433 / positive 932
test :  4,659 clips | negative 1,350 / neutral 1,025 / positive 2,284
Data match the v2 artifacts exactly.


## 3–4. Models and functions (copied unchanged from v2)

In [4]:
def masked_mean(x, mask):
    m = mask.unsqueeze(-1).float()
    return (x * m).sum(1) / m.sum(1).clamp(min=1)

class TextOnly(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(nn.Linear(DIMS["text"], D_MODEL), nn.ReLU(), nn.Dropout(0.2), nn.Linear(D_MODEL, 3))
    def forward(self, x, mask, avail, return_gate=False):
        out = self.net(masked_mean(x["text"], mask))
        return (out, None) if return_gate else out

class PaperRMSR(nn.Module):
    """Equation-level adaptation of EMO-FAR's RMSR (Eq. 10-14)."""
    def __init__(self):
        super().__init__()
        self.proj = nn.ModuleDict({m: nn.Linear(DIMS[m], D_MODEL) for m in MODS})          # Eq. 10
        self.u = nn.ParameterDict({m: nn.Parameter(torch.randn(D_MODEL) * 0.05) for m in MODS})  # Eq. 11
        self.Wq, self.Wk, self.Wv = (nn.Linear(D_MODEL, D_MODEL) for _ in range(3))
        self.cls = nn.Sequential(nn.Dropout(0.2), nn.Linear(D_MODEL, 3))
    def forward(self, x, mask, avail, return_gate=False):
        z = {m: self.proj[m](masked_mean(x[m], mask)) for m in MODS}
        q = torch.stack([torch.sigmoid((z[m] * self.u[m]).sum(-1)) for m in MODS], -1)   # Eq. 11
        gamma = q.softmax(-1)                                                             # Eq. 12
        att = {}
        for m in ["audio", "vision"]:                                                     # Eq. 13
            score = (self.Wq(z["text"]) * self.Wk(z[m])).sum(-1, keepdim=True) / D_MODEL ** 0.5
            att[m] = score.softmax(-1) * self.Wv(z[m])
        h = gamma[:, :1] * z["text"] + gamma[:, 1:2] * att["audio"] + gamma[:, 2:] * att["vision"]   # Eq. 14
        out = self.cls(h)
        return (out, gamma) if return_gate else out

class FixedFusion(nn.Module):
    """Token-level cross-attention + masked gating."""
    def __init__(self):
        super().__init__()
        self.proj = nn.ModuleDict({m: nn.Sequential(nn.Linear(DIMS[m], D_MODEL), nn.LayerNorm(D_MODEL)) for m in MODS})
        self.xattn = nn.ModuleDict({m: nn.MultiheadAttention(D_MODEL, 4, batch_first=True, dropout=0.1)
                                    for m in ["audio", "vision"]})
        self.gate = nn.Linear(3 * D_MODEL, 3)
        self.cls = nn.Sequential(nn.Dropout(0.2), nn.Linear(D_MODEL, D_MODEL), nn.ReLU(), nn.Linear(D_MODEL, 3))
    def forward(self, x, mask, avail, return_gate=False):
        p = {m: self.proj[m](x[m]) * avail[:, i, None, None] for i, m in enumerate(MODS)}
        h = {"text": masked_mean(p["text"], mask)}
        for m in ["audio", "vision"]:
            attended, _ = self.xattn[m](p["text"], p[m], p[m], key_padding_mask=~mask)
            h[m] = masked_mean(attended + p[m], mask)
        logits = self.gate(torch.cat([h[m] for m in MODS], -1))
        logits = logits.masked_fill(avail == 0, float("-inf"))
        w = logits.softmax(-1)
        fused = sum(w[:, i:i + 1] * h[m] for i, m in enumerate(MODS))
        out = self.cls(fused)
        return (out, w) if return_gate else out

MODELS = {"M1 Text-only": (TextOnly, 0.0),
          "M2 EMO-FAR RMSR (as written)": (PaperRMSR, 0.0),
          "M3 Fixed fusion": (FixedFusion, 0.0),
          "M4 Fixed fusion + modality dropout": (FixedFusion, MOD_DROPOUT)}
SHORT = {name: name.split()[0] for name in MODELS}                     # "M1" ... "M4"

In [5]:
from sklearn.metrics import f1_score, accuracy_score

def set_seed(s):
    random.seed(s); np.random.seed(s); torch.manual_seed(s); torch.cuda.manual_seed_all(s)

def sample_avail(n, rates, rng):
    """rates = (p_text, p_audio, p_vision): chance each modality is hidden. At least one stays."""
    avail = (rng.rand(n, 3) >= np.array(rates)).astype(np.float32)
    none_left = avail.sum(1) == 0
    avail[none_left, rng.randint(0, 3, none_left.sum())] = 1
    return avail

def to_batch(d, idx, avail):
    x = {m: torch.from_numpy(d["feats"][m][idx]).to(DEVICE) for m in MODS}
    for i, m in enumerate(MODS):
        x[m] = x[m] * torch.from_numpy(avail[:, i]).to(DEVICE)[:, None, None]
    return x, torch.from_numpy(d["mask"][idx]).to(DEVICE), torch.from_numpy(avail).to(DEVICE)

@torch.no_grad()
def predict(model, d, rates=(0, 0, 0), seed=0):
    model.eval(); rng = np.random.RandomState(seed); out = []
    for i in range(0, len(d["y"]), 256):
        idx = np.arange(i, min(i + 256, len(d["y"])))
        x, mask, avail = to_batch(d, idx, sample_avail(len(idx), rates, rng))
        out.append(model(x, mask, avail).softmax(-1).cpu().numpy())
    return np.concatenate(out)

def ece_score(probs, y, n_bins=15):
    conf, pred = probs.max(1), probs.argmax(1); ece = 0.0
    for lo, hi in zip(np.linspace(0, 1, n_bins + 1)[:-1], np.linspace(0, 1, n_bins + 1)[1:]):
        sel = (conf > lo) & (conf <= hi)
        if sel.any():
            ece += sel.mean() * abs((pred[sel] == y[sel]).mean() - conf[sel].mean())
    return float(ece)

def evaluate(probs, d):
    y, pred = d["y"], probs.argmax(1)
    nz = d["score"] != 0
    acc2 = ((probs[nz, 2] > probs[nz, 0]) == (d["score"][nz] > 0)).mean()
    return {"accuracy": accuracy_score(y, pred), "macro_f1": f1_score(y, pred, average="macro"),
            "acc2": float(acc2), "ece": ece_score(probs, y)}

def train(name, seed):
    cls, drop = MODELS[name]
    set_seed(seed)
    model = cls().to(DEVICE)
    opt = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=1e-4)
    counts = np.bincount(data["train"]["y"], minlength=3)
    class_w = torch.tensor(counts.sum() / (3 * counts), dtype=torch.float, device=DEVICE)
    rng = np.random.RandomState(seed); tr = data["train"]
    best, best_state, bad = -1, None, 0
    for ep in range(EPOCHS):
        model.train()
        order = rng.permutation(len(tr["y"]))
        for i in range(0, len(order), BATCH_SIZE):
            idx = order[i:i + BATCH_SIZE]
            x, mask, avail = to_batch(tr, idx, sample_avail(len(idx), (drop, drop, drop), rng))
            loss = F.cross_entropy(model(x, mask, avail), torch.from_numpy(tr["y"][idx]).to(DEVICE), weight=class_w)
            opt.zero_grad(); loss.backward(); torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0); opt.step()
        val = evaluate(predict(model, data["valid"]), data["valid"])["macro_f1"]
        if val > best:
            best, bad = val, 0
            best_state = {k: v.detach().clone() for k, v in model.state_dict().items()}
        else:
            bad += 1
            if bad >= PATIENCE:
                break
    model.load_state_dict(best_state)
    print(f"  {name:38s} seed {seed}: best valid macro-F1 = {best:.4f}  (stopped after epoch {ep + 1})")
    return model, float(best), ep + 1

def save_atomic(save_fn, path):
    """Write to a temporary file first, then rename: a disconnect never leaves a half-written file."""
    tmp = path + ".tmp"
    save_fn(tmp)
    os.replace(tmp, path)

## 5. Bootstrap helpers (as in Notebooks 4–5)

In [6]:
def video_of(ids):
    return np.array([str(i).split("$_$")[0] for i in ids])

def f1_from_counts(cm):
    cm = cm.reshape(cm.shape[:-1] + (3, 3)); tp = np.diagonal(cm, axis1=-2, axis2=-1)
    fp = cm.sum(-2) - tp; fn = cm.sum(-1) - tp; den = 2 * tp + fp + fn
    return np.where(den > 0, 2 * tp / np.maximum(den, 1e-12), 0).mean(-1)

y_test, y_val, y_tr = data["test"]["y"], data["valid"]["y"], data["train"]["y"]
_, vidx_test = np.unique(video_of(data["test"]["id"]), return_inverse=True); G_TEST = vidx_test.max() + 1
_r = np.random.RandomState(2026)
W_TEST = np.stack([np.bincount(_r.randint(0, G_TEST, G_TEST), minlength=G_TEST) for _ in range(N_BOOT)]).astype(np.float32)
SEED_DRAWS = np.random.RandomState(99).randint(0, len(SEEDS), (N_BOOT, len(SEEDS)))
mf1 = lambda y, p: f1_score(y, p, average="macro")

def video_counts(pred):
    c = np.zeros((G_TEST, 9)); np.add.at(c, (vidx_test, y_test * 3 + pred), 1); return c

def _p(b):
    return min(1.0, 2 * (min((b <= 0).sum(), (b >= 0).sum()) + 1) / (len(b) + 1))

def grouped_boot(preds_a, preds_b):
    per_seed = np.array([mf1(y_test, a) - mf1(y_test, b) for a, b in zip(preds_a, preds_b)])
    d = np.stack([f1_from_counts(W_TEST @ video_counts(a)) - f1_from_counts(W_TEST @ video_counts(b)) for a, b in zip(preds_a, preds_b)])
    sv = d[SEED_DRAWS, np.arange(N_BOOT)[:, None]].mean(1)
    return {"mean_diff_pp": 100 * per_seed.mean(), "per_seed_pp": ", ".join(f"{100*x:+.2f}" for x in per_seed),
            "seeds_positive": int((per_seed > 0).sum()), "sv_lo_pp": 100 * np.percentile(sv, 2.5), "sv_hi_pp": 100 * np.percentile(sv, 97.5),
            "sv_p": _p(sv)}

## 6. Extract M4's classifier input (fused vector) for every modality combination — saved per seed
Check: M4's own classifier applied to the extracted vector must give exactly M4's own output (test split).
The first seed is timed and a projection for the rest is printed.

In [7]:
@torch.no_grad()
def extract_fused(model, d, row, check=False):
    model.eval(); n = len(d["y"]); out = []
    for i in range(0, n, 512):
        idx = np.arange(i, min(i + 512, n))
        x, mask, av = to_batch(d, idx, np.tile(np.asarray(row, np.float32), (len(idx), 1)))
        p = {m: model.proj[m](x[m]) * av[:, j, None, None] for j, m in enumerate(MODS)}
        h = {"text": masked_mean(p["text"], mask)}
        for m in ["audio", "vision"]:
            att, _ = model.xattn[m](p["text"], p[m], p[m], key_padding_mask=~mask)
            h[m] = masked_mean(att + p[m], mask)
        w = model.gate(torch.cat([h[m] for m in MODS], -1)).masked_fill(av == 0, float("-inf")).softmax(-1)
        fused = sum(w[:, j:j + 1] * h[m] for j, m in enumerate(MODS))
        if check:
            assert torch.equal(model.cls(fused), model(x, mask, av)), "extraction does not reproduce M4's output"
        out.append(fused.numpy())
    return np.concatenate(out)

FUSED, M4_OWN = {}, {}
t0 = time.time()
for k, s in enumerate(SEEDS):
    f = f"{OUT_DIR}/reps/M4_seed{s}_fused.npz"
    m4 = FixedFusion(); m4.load_state_dict(torch.load(f"{CKPT_DIR}/M4_seed{s}.pt", map_location="cpu")); m4.eval()
    if os.path.exists(f):
        z = np.load(f); FUSED[s] = {c: {sp: z[f"{c}|{sp}"] for sp in ["train", "valid", "test"]} for c in CONDS}
        print(f"seed {s}: representations loaded from disk", flush=True)
    else:
        t1 = time.time()
        FUSED[s] = {c: {sp: extract_fused(m4, data[sp], ROW[c], check=(sp == "test")) for sp in ["train", "valid", "test"]} for c in CONDS}
        np.savez(f, **{f"{c}|{sp}": FUSED[s][c][sp] for c in CONDS for sp in ["train", "valid", "test"]})
        dt = time.time() - t1
        print(f"seed {s}: 21 extraction passes in {dt:.0f}s" + (f"  -> projected for all seeds: ~{dt * len(SEEDS) / 60:.1f} min" if k == 0 else ""), flush=True)
    with torch.no_grad():
        M4_OWN[s] = {c: m4.cls(torch.from_numpy(FUSED[s][c]["test"])).softmax(-1).numpy() for c in CONDS}
print(f"Extraction done ({time.time() - t0:.0f}s this session).")

seed 42: 21 extraction passes in 141s  -> projected for all seeds: ~7.1 min
seed 43: 21 extraction passes in 138s
seed 44: 21 extraction passes in 149s
Extraction done (432s this session).


## 7. Train the heads (H1–H4), 4 at a time, each saved as it finishes

In [8]:
class Head(nn.Module):
    """Same layers as M4's classifier: Dropout(0.2) -> Linear(128,128) -> ReLU -> Linear(128,3).
    With film=True: a per-combination scale and shift (initialised to zero) after the first Linear layer."""
    def __init__(self, film=False):
        super().__init__()
        self.drop, self.l1, self.l2 = nn.Dropout(0.2), nn.Linear(D_MODEL, D_MODEL), nn.Linear(D_MODEL, 3)
        self.film = film
        if film:
            self.gamma = nn.Parameter(torch.zeros(len(CONDS), D_MODEL)); self.beta = nn.Parameter(torch.zeros(len(CONDS), D_MODEL))
    def forward(self, x, c):
        h = self.l1(self.drop(x))
        if self.film:
            h = h * (1 + self.gamma[c]) + self.beta[c]
        return self.l2(torch.relu(h))

def train_head_job(seed, variant, cond, Xtr_np, Xva_np, Xte_np):
    """variant: 'H1' (M4's exposure), 'H2' (uniform), 'H3' (uniform + FiLM), 'H4' (one combination only).
    Same optimiser, learning rate, weight decay, batch size, class weights, clipping, epochs and patience as v2."""
    torch.set_num_threads(1); set_seed(seed)
    Xtr = torch.from_numpy(np.array(Xtr_np))   # private copy                                # (7, n, 128): one row per combination
    Xva = {c: torch.from_numpy(np.array(Xva_np[c])) for c in CONDS}
    head = Head(film=(variant == "H3")); opt = torch.optim.AdamW(head.parameters(), lr=LR, weight_decay=1e-4)
    counts = np.bincount(y_tr, minlength=3); cw = torch.tensor(counts.sum() / (3 * counts), dtype=torch.float)
    rng = np.random.RandomState(seed); yt = torch.from_numpy(y_tr); n = len(y_tr)
    val_conds = [cond] if variant == "H4" else CONDS
    best, best_state, bad, ep_done = -1, None, 0, 0
    for ep in range(EPOCHS):
        head.train(); order = rng.permutation(n)
        for i in range(0, n, BATCH_SIZE):
            idx = order[i:i + BATCH_SIZE]
            if variant == "H1":
                av = sample_avail(len(idx), (MOD_DROPOUT,) * 3, rng); ci = np.array([COND_OF_ROW[tuple(r)] for r in av])
            elif variant == "H4":
                ci = np.full(len(idx), CONDS.index(cond))
            else:
                ci = rng.randint(0, len(CONDS), len(idx))
            ci_t = torch.from_numpy(ci)
            loss = F.cross_entropy(head(Xtr[ci_t, torch.from_numpy(idx)], ci_t), yt[idx], weight=cw)
            opt.zero_grad(); loss.backward(); torch.nn.utils.clip_grad_norm_(head.parameters(), 1.0); opt.step()
        head.eval()
        with torch.no_grad():
            val = np.mean([mf1(y_val, head(Xva[c], torch.full((len(y_val),), CONDS.index(c))).argmax(1).numpy()) for c in val_conds])
        ep_done = ep + 1
        if val > best:
            best, bad = val, 0; best_state = {k: v.detach().clone() for k, v in head.state_dict().items()}
        else:
            bad += 1
            if bad >= PATIENCE:
                break
    head.load_state_dict(best_state); head.eval()
    with torch.no_grad():
        test = {c: head(torch.from_numpy(np.array(Xte_np[c])), torch.full((len(y_test),), CONDS.index(c))).softmax(-1).numpy()
                for c in ([cond] if variant == "H4" else CONDS)}
    return seed, variant, cond, best, ep_done, test

XTR = {s: np.stack([FUSED[s][c]["train"] for c in CONDS]) for s in SEEDS}       # passed to workers (memory-mapped by joblib)
JOBS = [(s, v, None) for v in ["H1", "H2", "H3"] for s in SEEDS] + [(s, "H4", c) for c in CONDS for s in SEEDS]
tag = lambda s, v, c: f"{v}_seed{s}" + (f"_{c.replace('+', '')}" if c else "")
todo = [j for j in JOBS if not os.path.exists(f"{OUT_DIR}/heads/{tag(*j)}.json")]
print(f"Heads: {len(JOBS)} in total, {len(JOBS) - len(todo)} already saved, {len(todo)} to train on {N_JOBS} cores", flush=True)
t0 = time.time()
for k in range(0, len(todo), N_JOBS):
    chunk = todo[k:k + N_JOBS]
    jobs = [delayed(train_head_job)(s_, v_, c_, XTR[s_], {cc: FUSED[s_][cc]["valid"] for cc in CONDS},
                                    {cc: FUSED[s_][cc]["test"] for cc in CONDS}) for s_, v_, c_ in chunk]
    for s, v, c, best, ep, test in Parallel(n_jobs=N_JOBS)(jobs):
        np.savez(f"{OUT_DIR}/heads/{tag(s, v, c)}_test_probs.npz", **{cc.replace("+", "p"): p for cc, p in test.items()})
        json.dump({"seed": s, "variant": v, "condition": c, "best_valid_macro_f1": best, "stopped_epoch": ep},
                  open(f"{OUT_DIR}/heads/{tag(s, v, c)}.json", "w"))
    done = len(JOBS) - len(todo) + k + len(chunk)
    el = time.time() - t0
    print(f"  heads saved: {done}/{len(JOBS)}  ({el:.0f}s elapsed" + (f", ~{el / (k + len(chunk)) * (len(todo) - k - len(chunk)) / 60:.1f} min left)" if k + len(chunk) < len(todo) else ")"), flush=True)
print("All heads trained.")

Heads: 30 in total, 0 already saved, 30 to train on 2 cores
  heads saved: 2/30  (20s elapsed, ~4.7 min left)
  heads saved: 4/30  (33s elapsed, ~3.6 min left)
  heads saved: 6/30  (41s elapsed, ~2.7 min left)
  heads saved: 8/30  (50s elapsed, ~2.3 min left)
  heads saved: 10/30  (59s elapsed, ~2.0 min left)
  heads saved: 12/30  (67s elapsed, ~1.7 min left)
  heads saved: 14/30  (78s elapsed, ~1.5 min left)
  heads saved: 16/30  (84s elapsed, ~1.2 min left)
  heads saved: 18/30  (91s elapsed, ~1.0 min left)
  heads saved: 20/30  (98s elapsed, ~0.8 min left)
  heads saved: 22/30  (103s elapsed, ~0.6 min left)
  heads saved: 24/30  (109s elapsed, ~0.5 min left)
  heads saved: 26/30  (114s elapsed, ~0.3 min left)
  heads saved: 28/30  (122s elapsed, ~0.1 min left)
  heads saved: 30/30  (126s elapsed)
All heads trained.


## 8. Results for every modality combination (test macro-F1, mean ± SD over 3 seeds)
References where they apply: M4 own, M2 own (EMO-FAR RMSR adaptation, same combination), M1 (text-only model; same prediction
whenever text is present), dedicated audio/video models from Notebook 4 (A, V, A+V), and the M1+U3 late fusion (T+A+V).

In [9]:
def load_head(s, v, c):
    z = np.load(f"{OUT_DIR}/heads/{tag(s, v, c)}_test_probs.npz"); return {k.replace("p", "+"): z[k] for k in z.files}

PRED = {}                                    # PRED[(method, cond)] -> list of predictions per seed
for c in CONDS:
    PRED[("M4 own", c)] = [M4_OWN[s][c].argmax(1) for s in SEEDS]
    for v in ["H1", "H2", "H3"]:
        PRED[(v, c)] = [load_head(s, v, None)[c].argmax(1) for s in SEEDS]
    PRED[("H4", c)] = [load_head(s, "H4", c)[c].argmax(1) for s in SEEDS]
# references
for s in SEEDS:
    m2 = PaperRMSR(); m2.load_state_dict(torch.load(f"{CKPT_DIR}/M2_seed{s}.pt", map_location="cpu")); m2.eval()
    with torch.no_grad():
        for c in CONDS:
            out = []
            for i in range(0, len(y_test), 512):
                idx = np.arange(i, min(i + 512, len(y_test)))
                x, mask, av = to_batch(data["test"], idx, np.tile(np.asarray(ROW[c], np.float32), (len(idx), 1)))
                out.append(m2(x, mask, av).argmax(1).numpy())
            PRED.setdefault(("M2 own", c), []).append(np.concatenate(out))
for c in CONDS:
    if "T" in c.split("+"):
        PRED[("M1 text-only", c)] = [np.load(f"{CKPT_DIR}/M1_seed{s}_test_probs.npy").argmax(1) for s in SEEDS]
for c, u in [("A", "U1"), ("V", "U2"), ("A+V", "U3")]:
    if all(os.path.exists(f"{V4_DIR}/checkpoints/{u}_seed{s}_test_probs.npy") for s in SEEDS):
        PRED[("Dedicated model", c)] = [np.load(f"{V4_DIR}/checkpoints/{u}_seed{s}_test_probs.npy").argmax(1) for s in SEEDS]
if all(os.path.exists(f"{V5_DIR}/LF_M1_U3_seed{s}_test_probs.npy") for s in SEEDS):
    PRED[("Late fusion M1+U3", "T+A+V")] = [np.load(f"{V5_DIR}/LF_M1_U3_seed{s}_test_probs.npy").argmax(1) for s in SEEDS]

rows = [{"method": m, "condition": c, "seed": s, "test_macro_f1": mf1(y_test, p)} for (m, c), ps in PRED.items() for s, p in zip(SEEDS, ps)]
res = pd.DataFrame(rows); res.to_csv(f"{OUT_DIR}/results_per_seed.csv", index=False)
METHODS = ["M2 own", "M4 own", "H1", "H2", "H3", "H4", "M1 text-only", "Dedicated model", "Late fusion M1+U3"]
tab = res.groupby(["method", "condition"]).test_macro_f1.agg(["mean", "std"])
summary = pd.DataFrame({c: [f"{100*tab.loc[(m, c), 'mean']:.2f} ± {100*tab.loc[(m, c), 'std']:.2f}" if (m, c) in tab.index else "—" for m in METHODS]
                        for c in CONDS}, index=METHODS)
summary.to_csv(f"{OUT_DIR}/results_summary.csv")
print("Test macro-F1 (%), mean ± SD over 3 seeds:"); summary

Test macro-F1 (%), mean ± SD over 3 seeds:


,T,A,V,A+V,T+A,T+V,T+A+V
M2 own,61.65 ± 0.24,31.62 ± 5.07,34.08 ± 3.36,36.05 ± 4.18,61.78 ± 0.37,62.35 ± 0.41,62.27 ± 0.53
M4 own,62.67 ± 0.38,40.02 ± 0.72,39.45 ± 1.17,41.76 ± 0.42,62.81 ± 0.34,63.57 ± 0.59,63.45 ± 0.50
H1,62.84 ± 0.19,40.11 ± 0.73,40.83 ± 0.04,42.80 ± 1.11,62.86 ± 0.37,63.80 ± 0.26,63.54 ± 0.33
H2,62.81 ± 0.59,39.86 ± 0.56,41.17 ± 0.14,43.14 ± 1.37,63.00 ± 0.68,63.88 ± 0.69,63.61 ± 0.73
H3,62.82 ± 0.11,39.95 ± 0.32,41.23 ± 0.61,43.38 ± 0.81,62.89 ± 0.36,63.70 ± 0.19,63.36 ± 0.71
H4,62.87 ± 0.20,40.61 ± 0.69,42.04 ± 0.83,43.77 ± 0.68,62.86 ± 0.24,63.81 ± 0.31,63.41 ± 0.82
M1 text-only,63.19 ± 0.37,—,—,—,63.19 ± 0.37,63.19 ± 0.37,63.19 ± 0.37
Dedicated model,—,41.06 ± 0.10,41.86 ± 0.78,43.74 ± 0.42,—,—,—
Late fusion M1+U3,—,—,—,—,—,—,63.86 ± 0.05


## 9. Pre-specified decision

In [10]:
comp_rows = []
for c in CONDS:
    for a, b in [("H4", "H2"), ("H2", "H1"), ("H3", "H2"), ("H1", "M4 own"), ("H2", "M4 own"), ("H4", "M4 own")]:
        comp_rows.append({"condition": c, "comparison": f"{a} − {b}", **grouped_boot(PRED[(a, c)], PRED[(b, c)])})
comp = pd.DataFrame(comp_rows); comp.to_csv(f"{OUT_DIR}/comparisons.csv", index=False)
pd.set_option("display.width", 250); pd.set_option("display.max_columns", 20)
print(comp[comp.condition.isin(["V", "A+V", "T+A+V"])].round(3).to_string(index=False))

sig_pos = lambda r: r["mean_diff_pp"] > MARGIN_PP and r["sv_lo_pp"] > 0
cell = lambda c, q: comp[(comp.condition == c) & (comp.comparison == q)].iloc[0]
conditional = any(sig_pos(cell(c, "H4 − H2")) for c in ["V", "A+V"])
guard = {v: tab.loc[(v, "T+A+V"), "mean"] - tab.loc[("M4 own", "T+A+V"), "mean"] for v in ["H1", "H2", "H3", "H4"]}
flagged = [v for v, d in guard.items() if 100 * d < -MARGIN_PP]
if conditional:
    verdict = ("CONDITIONAL DECISION BOUNDARIES SUPPORTED: separate heads beat a shared head with equal exposure by > 1 pp "
               "(significant). A conditional-classifier direction is justified; compare with missing-aware prompts, MPLMM and "
               "Flex-MoE before any novelty claim; confirm on MOSI.")
else:
    verdict = ("EXPOSURE EXPLAINS IT: with equal exposure, one shared head is within the 1 pp margin of separate heads (or the "
               "difference is not significant). No new classifier mechanism is justified by this evidence; report as an analysis "
               "finding (close to MCE's availability reweighting); confirm on MOSI.")
secondary = {c: {q: round(float(cell(c, q)["mean_diff_pp"]), 3) for q in ["H2 − H1", "H3 − H2", "H4 − H2"]} for c in ["V", "A+V"]}
out = {"verdict": verdict, "full_condition_change_vs_M4_pp": {v: round(100 * d, 3) for v, d in guard.items()},
       "flagged_full_condition_loss": flagged, "secondary_pp": secondary}
json.dump(out, open(f"{OUT_DIR}/DECISION.json", "w"), indent=2)
print("\nRESEARCH DECISION:", verdict)
print("Full condition (T+A+V) change vs M4 own, pp:", out["full_condition_change_vs_M4_pp"], "| flagged:", flagged or "none")
print("Secondary (pp):", secondary)

condition  comparison  mean_diff_pp         per_seed_pp  seeds_positive  sv_lo_pp  sv_hi_pp  sv_p
        V     H4 − H2         0.877 -0.01, +1.74, +0.90               2    -0.366     2.374 0.177
        V     H2 − H1         0.339 +0.37, +0.17, +0.48               3    -0.680     1.351 0.508
        V     H3 − H2         0.069 -0.22, -0.20, +0.63               1    -0.731     0.787 0.809
        V H1 − M4 own         1.377 +2.59, +1.20, +0.34               3    -0.024     2.919 0.055
        V H2 − M4 own         1.716 +2.96, +1.37, +0.81               3     0.103     3.356 0.037
        V H4 − M4 own         2.593 +2.95, +3.11, +1.72               3     1.120     4.096 0.002
      A+V     H4 − H2         0.627 +1.48, +0.39, +0.01               3    -0.539     1.844 0.278
      A+V     H2 − H1         0.340 -0.65, +1.19, +0.48               2    -0.859     1.596 0.579
      A+V     H3 − H2         0.243 +0.71, +0.44, -0.42               2    -0.549     1.154 0.567
      A+V H1 − M4 ow

## 10. Files written (earlier folders untouched)

In [11]:
for root, _, files in os.walk(OUT_DIR):
    if root.endswith("heads") or root.endswith("reps"):
        print(f"   {os.path.relpath(root, OUT_DIR)}/  ({len(files)} files)"); continue
    for fn in sorted(files):
        print("  ", os.path.relpath(os.path.join(root, fn), OUT_DIR))

   DECISION.json
   comparisons.csv
   results_per_seed.csv
   results_summary.csv
   run_info.json
   test_ids.csv
   test_labels.npy
   test_scores.npy
   valid_labels.npy
   reps/  (3 files)
   heads/  (60 files)
